In [1]:
%%bash

wget -O "./protein/Apis_cerana.faa.gz" "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/029/169/275/GCF_029169275.1_AcerK_1.0/GCF_029169275.1_AcerK_1.0_protein.faa.gz"
gunzip "./protein/Apis_cerana.faa.gz"

--2026-11-01 10:21:08--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/029/169/275/GCF_029169275.1_AcerK_1.0/GCF_029169275.1_AcerK_1.0_protein.faa.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.31, 130.14.250.7, 2607:f220:41e:250::7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6361798 (6.1M) [application/x-gzip]
Saving to: ‘./protein/Apis_cerana.faa.gz’

     0K .......... .......... .......... .......... ..........  0% 89.5K 69s
    50K .......... .......... .......... .......... ..........  1% 34.2M 34s
   100K .......... .......... .......... .......... ..........  2%  180K 34s
   150K .......... .......... .......... .......... ..........  3%  105M 25s
   200K .......... .......... .......... .......... ..........  4%  182K 27s
   250K .......... .......... .......... .......... ..........  4% 35.1M 22s
   300K .......... .......... .......... .......... 

确定 adata 的 gene_id 与蛋白组中的 header 都是 GTF 中特定列的子集：

In [3]:
import scanpy as sc
import pandas as pd
from pyfaidx import Fasta

In [ ]:
Acer_gtf = pd.read_csv("./metadata/Apis_cerana.tsv", sep="\t") # GCF_029169275.1, not latest GCF_029169275.2!
# Acer_gtf
#! Apis_cerana.tsv: see https://github.com/zliu90693/anno_cells/tree/main/.subprojects/MetaNeighbor/metadata/Apis_cerana.tsv

/tmp/ipykernel_307261/1468964168.py:1: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  Acer_gtf = pd.read_csv("./metadata/Apis_cerana.tsv", sep="\t") # GCF_029169275.1, not latest GCF_029169275.2!


In [7]:
Acer_gtf_gid = set(Acer_gtf["gene_id"])
Acer_gtf_pid = set(Acer_gtf["protein_id"])

In [8]:
Acer_adata = sc.read_h5ad("./matrix/01_Acer_transfromHY.h5ad")

In [11]:
(Acer_adata.var.index != Acer_adata.var["gene_ids"]).sum()

0

In [12]:
Acer_adata_gid = set(Acer_adata.var["gene_ids"])

In [ ]:
Acer_protein = Fasta("./protein/Apis_cerana.faa")

In [15]:
Acer_protein_pid = set(header for header in Acer_protein.keys())

In [20]:
print(len(Acer_gtf_gid), len(Acer_adata_gid), len(Acer_gtf_gid & Acer_adata_gid))


12561 11817 11817


In [21]:
print(len(Acer_gtf_pid), len(Acer_protein_pid), len(Acer_gtf_pid & Acer_protein_pid))

29189 29188 29188


构建 protein-gene 对应关系

由于 "./protein/Apis_cerana.faa" 的格式不符合 Ensembl 的 gene_symbol: header 结构，因此不使用 SATURN 中的 map_gene_symbol_to_protein_ids.py 处理 NCBI faa，手动处理：

In [22]:
import json
import pandas as pd

Acer_gtf = pd.read_csv("metadata/Apis_cerana.tsv", sep="\t", low_memory=False)

m = Acer_gtf.dropna(subset=["gene_id", "protein_id"])[["gene_id", "protein_id"]]
m = m.drop_duplicates()
m["gene_id"] = m["gene_id"].astype(str)
m["protein_id"] = m["protein_id"].astype(str)

gene_to_proteins = {}
for gene, sub in m.groupby("gene_id"):
    gene_to_proteins[gene] = sorted(set(sub["protein_id"]))

with open("./gene_protein_map/Acer.gene_id_to_protein_ID.json", "w") as f:
    json.dump(gene_to_proteins, f, indent=4, sort_keys=True)